In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from modules import data_loaders, learning
from modules.models import LMNet



In [11]:
ptb_path = "Data/ptb/"
fnames = ["ptb.char.train.txt", "ptb.char.valid.txt", "ptb.char.test.txt"]

# config = sys.argv[1] if len(sys.argv) > 1 else "LSTM"
# config = 'LCCL'
config = 'LCRL'
file_name = f"Results/weights_{config}"

# n_hidden = 256
n_hidden = 3
seq_len = 35
grad_clip = 10
batch_size = 32
vocab_size = 10000
learning_rate = 0.002
hid_prop = True
num_epochs = 10
save_fq = 50
print_fq = 1
seed = 0



In [4]:
train_data, valid_data, test_data, word2idx, idx2word = data_loaders.load_PTB_word(ptb_path, *fnames)
train_data = data_loaders.PTB_word(train_data, seq_len, batch_size)
valid_data = data_loaders.PTB_word(valid_data, seq_len, batch_size)
test_data = data_loaders.PTB_word(test_data, seq_len, batch_size)

train_loader = data_loaders.PTBLoader(train_data, vocab_size)
valid_loader = data_loaders.PTBLoader(valid_data, vocab_size)
test_loader = data_loaders.PTBLoader(test_data, vocab_size)

vocab_size = 10000


In [5]:
def ce_loss_fn(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    loss_fn = nn.CrossEntropyLoss()
    y = torch.remainder(y, x.size(-1)).view(-1)
    x = x.view(-1, x.size(-1))
    return loss_fn(x, y)

In [16]:
lm_net = LMNet(vocab_size, n_hidden, config, hid_prop, batch_size)

criterion = ce_loss_fn
optimizer = optim.Adam(lm_net.parameters(), lr=learning_rate)

lm_trainer = learning.LMTrainer(lm_net,
                                optimizer,
                                criterion=criterion,
                                num_epochs=num_epochs,
                                train_loader=train_loader,
                                valid_loader=valid_loader,
                                test_loader=test_loader)

lm_trainer.train()


epoch 0001/10 | batch 0000/830 | base_loss 9.2092 | total_loss 9.1076 | tokens 1120 


KeyboardInterrupt: 

In [ ]:

def generate_text(model, seed_words, word2idx, idx2word, vocab_size, max_len=30, device='cpu'):
    model.eval()
    with torch.no_grad():
        # 1️⃣ Превращаем слова в индексы
        idx_seq = [word2idx[w] for w in seed_words]
        x = torch.tensor([idx_seq], dtype=torch.long, device=device)

        # 2️⃣ One-hot кодировка → (1, seq_len, vocab_size)
        x_onehot = nn.functional.one_hot(x, num_classes=vocab_size).float()

        # 3️⃣ Генерация
        for _ in range(max_len):
            # Прогон через модель
            logits = model(x_onehot)

            # Берём логиты последнего шага
            next_logits = logits[:, -1, :]  # (1, vocab_size)
            probs = nn.functional.softmax(next_logits, dim=-1).squeeze(0)

            # Сэмплируем следующее слово
            next_idx = torch.multinomial(probs, num_samples=1).item()

            # Добавляем новое слово в последовательность
            x = torch.cat([x, torch.tensor([[next_idx]], device=device)], dim=1)

            # Пересчитываем one-hot
            x_onehot = nn.functional.one_hot(x, num_classes=vocab_size).float()

        # 4️⃣ Конвертируем индексы обратно в слова
        generated_words = [idx2word[int(i)] for i in x[0].tolist()]
        return " ".join(generated_words)

seed = ["asbestos"]
print(generate_text(lm_net, seed, word2idx, idx2word, vocab_size=len(word2idx), max_len=50))


asbestos N the jumbo there around N <eos> to hang in 's far simply mr. corp in although by data <eos> problem <eos> $ the the automotive bonds and the properly <eos> but lobbyist were <unk> enough proposed actor million for someone all businesses broadcasting <unk> <eos> shares in from large


profiling

In [22]:
import torch.profiler as profiler


def test_with_loader(model, loader):
    model.train()
    # -------------------
    # профилируем одну эпоху
    # -------------------
    with profiler.profile(
        activities=[profiler.ProfilerActivity.CPU],
        record_shapes=True,
        with_stack=True
    ) as prof:
        for step, (x, y) in enumerate(loader):
            with profiler.record_function("model_inference"):
                logits = model(x)
            if step >= 2:
                break
    print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=15))


model = LMNet(vocab_size, n_hidden, config, hid_prop, batch_size)

test_with_loader(model, train_loader)


-----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                         Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
              model_inference        13.37%      36.403ms        77.70%     211.595ms      70.532ms             3  
                 aten::matmul         0.19%     507.800us        24.73%      67.353ms     606.779us           111  
                     aten::mm        24.45%      66.588ms        24.47%      66.630ms     600.269us           111  
                    aten::mul        17.09%      46.549ms        18.31%      49.856ms      38.380us          1299  
                aten::one_hot         0.06%     166.500us        12.54%      34.148ms      11.383ms             3  
                  aten::zeros         0.01%      35.500us        12.25% 